Obtain ID's of records that have zero subjects.  
Then free to experiment with methods to enrich their metadata.

In [1]:
NO_RECORDS = 150

In [2]:
from datetime import datetime, timezone
from json import JSONDecodeError
from logging.config import dictConfig
from typing import Any, Optional
import psycopg
from psycopg import errors as psycopg_errors
from psycopg.rows import dict_row
from config.logging_config import LOGGING_CONFIG
from config.postgres_config import PostgresConfig
from tasks import transform_batch, add_file_metadata
import os
from fastapi import FastAPI, Query, HTTPException
import logging
from pydantic import BaseModel, Field
from utils.queue_utils import HarvestEventQueue, detect_identifier_type

from lxml import etree as ET
# API calls
import requests


# added for these experiments
from utils.enrichment_methods import get_doi, enrich_subjects_from_openaire, merge_subjects_from_new_method

d:\eosc_dc\metadata-warehouse\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dictConfig(LOGGING_CONFIG)
logger = logging.getLogger(__name__)

In [4]:
user = os.getenv("POSTGRES_USER")
passwd = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_ADDRESS", "127.0.0.1")


if host == '':
    host = "127.0.0.1"
    user = os.getenv("POSTGRES_ADMIN")

connection_params = {
    "user" : user,
    "password" : passwd,
    "dbname" : "datasetdb",
    "port" : 5432,
    "host" : host
}

In [5]:
query_records = """
            -- Exploring subjects from record raw metadata

-- Every xpath defines a potential XML structure with corresponding subject tag placement.
-- If raw_metadata conforms to ANY of these xpaths, subjects will be concatenated <subject> tags. (removing array_to_string() will return arrays)
-- If raw_metadata doesn't conform to any path (and probably has no subject tags), subjects will be ''


WITH query AS (
    SELECT
        records.*,
        array_to_string(
                xpath('oai:record/oai:metadata/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oaidc:oai_datacite/oaidc:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/},
                      {oaidc, http://schema.datacite.org/oai/oai-1.1/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oai:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
				-- note the 3 in datacite
                    '{{datacite, http://datacite.org/schema/kernel-3},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[] ||
				-- same path as previous but uses kernel-4 namespace
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[]
				, ', '
        ) AS subjects
    FROM public.records
)
SELECT 
    id,
    title,
    raw_metadata,
    subjects
FROM query
WHERE query.subjects = ''
LIMIT %s
;
"""

In [6]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        cur = conn.cursor()
        
        cur.execute(query_records, (NO_RECORDS,))
        
        results = cur.fetchall()
        print(results)

        record_ids = [str(row["id"]) for row in results]
        record_titles = [str(row["title"]) for row in  results]
        record_raw_metadata = [str(row["raw_metadata"]) for row in  results]


except Exception as e:
    logger.exception("Failed to fetch harvest runs")
    raise HTTPException(status_code=500, detail=str(e))

[{'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::599', 'title': 'Data for: Electron-phonon coupling in n-type Ge two-dimensional systems', 'raw_metadata': '<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:rodare.hzdr.de:599</identifier>\n    <datestamp>2024-08-08T10:40:44Z</datestamp>\n    <setSpec>openaire_data</setSpec>\n    <setSpec>user-rodare</setSpec>\n    <setSpec>user-elbe</setSpec>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.0/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.0/ oai_datacite.xsd">\n      <isReferenceQuality>true</isReferenceQuality>\n      <schemaVersion>4.1</schemaVersion>\n      <datacentreSymbol>CERN.ZENODO</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.1/metadata.xsd"

In [7]:
record_titles

['Data for: Electron-phonon coupling in n-type Ge two-dimensional systems',
 'Dataset and scripts for A Deep Dive into Machine Learning Density Functional Theory for Materials Science and Chemistry',
 'Phenix EOL natural convection test: Serpent/DYN3D/ATHLET I/O data',
 'Data publication: Generalized density functional theory framework for the non-linear density response of quantum many-body systems',
 'Data publication: Spectral Deconvolution without the Deconvolution: Extracting Temperature from X-ray Thomson Scattering Spectra without the Source-and-Instrument Function',
 'Data publication: Application of a spherically averaged pair potential in ab initio path integral Monte Carlo simulations of the warm dense electron gas',
 'Data for: Bonding Trends in Tetravalent Th–Pu Monosalen Complexes',
 'Coordination of Trivalent Lanthanum and Cerium, and Tetravalent Cerium and Actinides (An = Th(IV), U(IV), Np(IV)) by a 4-Phosphoryl 1H-Pyrazol-5-olate Ligand in Solution and the Solid State'

In [8]:
record_raw_metadata[2]

'<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:rodare.hzdr.de:2007</identifier>\n    <datestamp>2022-12-09T14:53:29Z</datestamp>\n    <setSpec>openaire_data</setSpec>\n    <setSpec>user-energy</setSpec>\n    <setSpec>user-rodare</setSpec>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.0/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.0/ oai_datacite.xsd">\n      <isReferenceQuality>true</isReferenceQuality>\n      <schemaVersion>4.1</schemaVersion>\n      <datacentreSymbol>CERN.ZENODO</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.1/metadata.xsd">\n          <identifier identifierType="DOI">10.14278/rodare.2007</identifier>\n          <creators>\n            <creator>\n              <creatorNa

In [9]:
get_doi(record_raw_metadata[2])

'10.14278/rodare.2007'

# DOI check through bigger repositories  (#1)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Seaches thru OpenAlex or Openaire - apis?
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword

OpenAIRE API
- they're officialy moving to a Graph API in May 2026
https://api.openaire.eu/graph/swagger-ui/index.html

In [10]:
import logging
logging.getLogger("urllib3").setLevel(logging.WARNING)

In [11]:
doi = get_doi(record_raw_metadata[0]) 

if not doi:
    print("NO DOI")
else:
    print(enrich_subjects_from_openaire(doi))

[]


In [12]:
# get only records that have a DOI
ix_has_doi = []

for i in range(0, len(record_raw_metadata)):
    rec = get_doi(record_raw_metadata[i])
    if rec:
        ix_has_doi.append(i)

print(ix_has_doi)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 72, 74, 76, 84, 85, 86, 87, 93, 96, 98, 102, 103, 105, 106, 107, 109, 110, 120, 140, 146]


In [13]:
id_to_enrich = [record_ids[i] for i in ix_has_doi]
doi_to_enrich = [get_doi(record_raw_metadata[i]) for i in ix_has_doi]

print(id_to_enrich)
print(doi_to_enrich)

['a64cfb20-e461-4ec0-a069-a00c1fde6c64::599', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1294', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::2007', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::4487', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::4533', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::3959', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::586', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::644', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1282', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1848', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::342', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::2268', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::344', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::2608', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::2146', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::994', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1702', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::2309', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::4296', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1856', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1479', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::

In [14]:
# limit to smaller section
#id_to_enrich = id_to_enrich[:30]
#doi_to_enrich = doi_to_enrich[:30]

In [15]:
from utils.enrichment_methods import _openaire_batch_doi_resolve, _opeanaire_get_doi_subjects, connect_ids_with_subjects_by_doi, batch_enrich_subjects_from_openaire

api_result = _openaire_batch_doi_resolve(doi_to_enrich)

print(api_result[:3])

[{'authors': [{'fullName': 'Maalberg, Andrei', 'name': 'Andrei', 'surname': 'Maalberg', 'rank': 1, 'pid': {'id': {'scheme': 'orcid_pending', 'value': '0000-0001-9560-0487'}, 'provenance': None}}, {'fullName': 'Kuntzsch, Michael', 'name': 'Michael', 'surname': 'Kuntzsch', 'rank': 2, 'pid': {'id': {'scheme': 'orcid_pending', 'value': '0000-0002-8145-5837'}, 'provenance': None}}, {'fullName': 'Petlenkov, Eduard', 'name': 'Eduard', 'surname': 'Petlenkov', 'rank': 3, 'pid': {'id': {'scheme': 'orcid_pending', 'value': '0000-0003-2167-6280'}, 'provenance': None}}], 'openAccessColor': None, 'publiclyFunded': None, 'type': 'dataset', 'language': {'code': 'und', 'label': 'Undetermined'}, 'countries': None, 'subjects': None, 'mainTitle': 'Data publication: FPGA Testbench for Beam-Based Feedback', 'subTitle': None, 'descriptions': ['The testbench data to compare\xa0the time response of a 4th-order regulator implemented in fixed-point with the same regulator simulated in a floating-point MATLAB mod

In [16]:
doi_subjects = _opeanaire_get_doi_subjects(api_result)
doi_subjects

[{'doi': '10.14278/rodare.327', 'subjects': ['6. Clean water']},
 {'doi': '10.14278/rodare.328', 'subjects': ['6. Clean water']},
 {'doi': '10.14278/rodare.2246', 'subjects': ['7. Clean energy']},
 {'doi': '10.14278/rodare.2245', 'subjects': ['7. Clean energy']},
 {'doi': '10.5286/isis.e.rb1835020',
  'subjects': ['STFC ISIS Neutron and Muon Data']},
 {'doi': '10.5286/isis.e.rb1835020-1',
  'subjects': ['STFC ISIS Neutron and Muon Data']}]

In [17]:
id_dois = []
for ix in range(len(id_to_enrich)):
    id_dois.append(
        {
            'id': id_to_enrich[ix],
            'doi': doi_to_enrich[ix].lower(),
        }
    )
id_dois

[{'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::599',
  'doi': '10.14278/rodare.599'},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1294',
  'doi': '10.14278/rodare.1294'},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::2007',
  'doi': '10.14278/rodare.2007'},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::4487',
  'doi': '10.14278/rodare.4487'},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::4533',
  'doi': '10.14278/rodare.4533'},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::3959',
  'doi': '10.14278/rodare.3959'},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::586',
  'doi': '10.14278/rodare.586'},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::644',
  'doi': '10.14278/rodare.644'},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1282',
  'doi': '10.14278/rodare.1282'},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1848',
  'doi': '10.14278/rodare.1848'},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::342',
  'doi': '10.14278/rodare.342'},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fd

In [18]:
enriched_list = connect_ids_with_subjects_by_doi(id_dois, doi_subjects)
enriched_list

[{'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::599',
  'doi': '10.14278/rodare.599',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1294',
  'doi': '10.14278/rodare.1294',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::2007',
  'doi': '10.14278/rodare.2007',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::4487',
  'doi': '10.14278/rodare.4487',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::4533',
  'doi': '10.14278/rodare.4533',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::3959',
  'doi': '10.14278/rodare.3959',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::586',
  'doi': '10.14278/rodare.586',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::644',
  'doi': '10.14278/rodare.644',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1282',
  'doi':

In [19]:
# Only succesfully enriched IDs and subjects.
# Could be useful for comparing how many ids we sent to be enriched vs
# how many actually got new subjects
[el for el in enriched_list if el['enriched_subjects'] is not None]

[{'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::599',
  'doi': '10.14278/rodare.599',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1294',
  'doi': '10.14278/rodare.1294',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::2007',
  'doi': '10.14278/rodare.2007',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::4487',
  'doi': '10.14278/rodare.4487',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::4533',
  'doi': '10.14278/rodare.4533',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::3959',
  'doi': '10.14278/rodare.3959',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::586',
  'doi': '10.14278/rodare.586',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::644',
  'doi': '10.14278/rodare.644',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1282',
  'doi':

In [20]:
# All-in-one function

batch_enrich_subjects_from_openaire(id_dois)

[{'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::599',
  'doi': '10.14278/rodare.599',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1294',
  'doi': '10.14278/rodare.1294',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::2007',
  'doi': '10.14278/rodare.2007',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::4487',
  'doi': '10.14278/rodare.4487',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::4533',
  'doi': '10.14278/rodare.4533',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::3959',
  'doi': '10.14278/rodare.3959',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::586',
  'doi': '10.14278/rodare.586',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::644',
  'doi': '10.14278/rodare.644',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1282',
  'doi':

In [21]:
# Comparison: Number of initial records vs number of records that have gained subjects via this method
enriched = batch_enrich_subjects_from_openaire(id_dois)
print(len([el['doi'] for el in enriched]))
print(len([el['doi'] for el in enriched if el['enriched_subjects'] is not None]))

86
86


In [22]:
# Taking a look at the metadata initially obtained from DB, subset of few records
full_info_enriched = [el for el in enriched if el['enriched_subjects'] is not None]

first_enriched = full_info_enriched[:10]

In [23]:
# indexes that have been enriched
ix_original = [record_ids.index(rec['id']) for rec in first_enriched]
ix_original

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [24]:
first_enriched

[{'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::599',
  'doi': '10.14278/rodare.599',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1294',
  'doi': '10.14278/rodare.1294',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::2007',
  'doi': '10.14278/rodare.2007',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::4487',
  'doi': '10.14278/rodare.4487',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::4533',
  'doi': '10.14278/rodare.4533',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::3959',
  'doi': '10.14278/rodare.3959',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::586',
  'doi': '10.14278/rodare.586',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::644',
  'doi': '10.14278/rodare.644',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1282',
  'doi':

In [25]:
[record_raw_metadata[i] for i in ix_original]

['<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:rodare.hzdr.de:599</identifier>\n    <datestamp>2024-08-08T10:40:44Z</datestamp>\n    <setSpec>openaire_data</setSpec>\n    <setSpec>user-rodare</setSpec>\n    <setSpec>user-elbe</setSpec>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.0/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.0/ oai_datacite.xsd">\n      <isReferenceQuality>true</isReferenceQuality>\n      <schemaVersion>4.1</schemaVersion>\n      <datacentreSymbol>CERN.ZENODO</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.1/metadata.xsd">\n          <identifier identifierType="DOI">10.14278/rodare.599</identifier>\n          <creators>\n            <creator>\n              <creatorName>

## OpenAlex API
https://developers.openalex.org/
https://developers.openalex.org/guides/authentication

Get multiple works by DOI

use | between DOIs, up to 100 DOIs at once!!
check out how the results look then.

https://api.openalex.org/works?filter=doi:https://doi.org/10.1234/a|https://doi.org/10.1234/b|https://doi.org/10.1234/c&per_page=100



Use the select parameter to return only specific fields:
https://api.openalex.org/works?select=id,doi,display_name

In [26]:
OpenAlex_API_key = os.getenv("OPENALEX_API_KEY")

In [27]:
OPENALEX_API_URL = "https://api.openalex.org/works/"

doi = '10.15151/esrf-es-1473312981'

# We have to append the ID directly in the path
query_url = OPENALEX_API_URL + "https://doi.org/" + doi

response = requests.get(
    query_url,
    params={"api_key": OpenAlex_API_key}, 
    timeout=30
)

data = response.json()


In [28]:
oa_DOIS = [
    '10.15151/esrf-es-1001518064',
    '10.15151/esrf-es-2323561777',
    '10.15151/esrf-es-2323561778',
    '10.15151/esrf-es-2323561877',
    'https://doi.org/10.5286/ISIS.E.100051344',
    '10.34894/AUUSS2',
    '10.17026/DANS-28H-C53V',
    '10.34894/VLSN1K',
    '10.15151/ESRF-ES-1030397492',
    '10.17026/AR/DMHU53',
    '10.17026/AR/GHYQTO',
    '10.17026/DANS-ZU2-HF3N',
    '10.17026/DANS-ZTM-7SA5',
    '10.17026/DANS-ZUK-HN2Z',
    'https://doi.org/10.1016/j.paid.2021.110888',
    '10.23662/fors-ds-686-1',
    '10.1056/NEJMoa2034577',
    '10.1056/NEJMoa2021436',
    '10.1056/NEJMoa1511939',
    '10.1056/NEJMoa1800389'
]

the API result object can have a result for a given DOI, but that result doesn't contain any keywords = doesn't contribute to enrichment
Hence the len(api_result) can be larger than len(enriched_dois)

Additionaly if we set a high keyword score threshold, a certain result MIGHT have keywords but none are relevant enough - that will also not contribute to enrichment

In [29]:
from utils.enrichment_methods import batch_enrich_subjects_from_openalex

enriched_results = batch_enrich_subjects_from_openalex(
    id_dois,
    OpenAlex_API_key,
    score_treshold = 0.6
)

enriched_results = [el for el in enriched_results if el['enriched_subjects']]
print(len(enriched_results))
enriched_results

16


[{'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::599',
  'doi': '10.14278/rodare.599',
  'enriched_subjects': {'Coupling (piping)',
   'Semiconductor Quantum Structures and Devices',
   'Semiconductor materials and interfaces'}},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::4533',
  'doi': '10.14278/rodare.4533',
  'enriched_subjects': {'Deconvolution', 'Thomson scattering'}},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::586',
  'doi': '10.14278/rodare.586',
  'enriched_subjects': {'Organometallic Complex Synthesis and Catalysis',
   'Radioactive element chemistry and processing'}},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::4296',
  'doi': '10.14278/rodare.4296',
  'enriched_subjects': {'Thomson scattering'}},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1063',
  'doi': '10.14278/rodare.1063',
  'enriched_subjects': {'Absorption (acoustics)',
   'Analytical Chemistry and Sensors',
   'Nonlinear Optical Materials Studies',
   'Spectroscopy Techniques in Biomedical and Chemical 

In [30]:
# Showcase  - manually input DOIs

test_dois = [
    {'id': "testid1",
     'doi': '10.1016/j.paid.2021.110888'},
    {'id': "testid2",
     'doi': '10.23662/fors-ds-686-1'}
]

enriched_results = batch_enrich_subjects_from_openalex(
    test_dois,
    OpenAlex_API_key,
    score_treshold = 0.6
)

enriched_results

[{'id': 'testid1',
  'doi': '10.1016/j.paid.2021.110888',
  'enriched_subjects': {'Conscientiousness',
   'Extraversion and introversion',
   'Flow Experience in Various Fields',
   'Mind wandering and attention',
   'Neuroticism',
   'Openness to experience',
   'Psychology',
   'Sport Psychology and Performance'}},
 {'id': 'testid2',
  'doi': '10.23662/fors-ds-686-1',
  'enriched_subjects': set()}]

In [31]:
# Compare openalex with openaire:
oaire_results = batch_enrich_subjects_from_openaire(test_dois)
oaire_results

[{'id': 'testid1',
  'doi': '10.1016/j.paid.2021.110888',
  'enriched_subjects': {'05 social sciences',
   '0501 psychology and cognitive sciences',
   '10. No inequality'}},
 {'id': 'testid2',
  'doi': '10.23662/fors-ds-686-1',
  'enriched_subjects': {'10. No inequality',
   '3. Good health',
   '4. Education',
   '8. Economic growth'}}]

In [32]:
merge_subjects_from_new_method(enriched_results, oaire_results)

[{'id': 'testid1',
  'doi': '10.1016/j.paid.2021.110888',
  'enriched_subjects': {'05 social sciences',
   '0501 psychology and cognitive sciences',
   '10. No inequality',
   'Conscientiousness',
   'Extraversion and introversion',
   'Flow Experience in Various Fields',
   'Mind wandering and attention',
   'Neuroticism',
   'Openness to experience',
   'Psychology',
   'Sport Psychology and Performance'}},
 {'id': 'testid2',
  'doi': '10.23662/fors-ds-686-1',
  'enriched_subjects': {'10. No inequality',
   '3. Good health',
   '4. Education',
   '8. Economic growth'}}]

# DOI resolver method  (#2)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Talks to an API such as DataCite or SignPosting.
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword